In [2]:
from backtesting import Backtest, Strategy
from backtesting.lib import crossover
import talib
import yfinance as yf

In [3]:
gc = yf.Ticker('GC=F')
data = gc.history('max')

In [4]:
# Calculate the split point
split_idx = int(len(data) * 0.8)

# Split the data
iss_data = data.iloc[:split_idx]
oss_data = data.iloc[split_idx:]

print(f"ISS Range: {iss_data.index[0]} to {iss_data.index[-1]}")
print(f"OSS Range: {oss_data.index[0]} to {oss_data.index[-1]}")

ISS Range: 2000-08-30 00:00:00-04:00 to 2021-03-11 00:00:00-05:00
OSS Range: 2021-03-12 00:00:00-05:00 to 2026-04-23 00:00:00-04:00


In [5]:
class EMA(Strategy):

    fast_window = 10
    slow_window = 30
    trend_filter = 65

    def init(self):
        self.fast_ema = self.I(talib.EMA, self.data.Close, self.fast_window)
        self.slow_ema = self.I(talib.EMA, self.data.Close, self.slow_window)
        self.filter_ema = self.I(talib.EMA, self.data.Close, self.trend_filter)

    def next(self):
        price = self.data.Close[-1]

        if crossover(self.fast_ema, self.slow_ema) and price>self.filter_ema:
            self.buy()

        elif crossover(self.slow_ema, self.fast_ema):
            self.position.close()

In [7]:
bt = Backtest(iss_data, EMA, cash=10_000)

In [8]:
stats = bt.run()

Backtest.run:   0%|          | 0/5083 [00:00<?, ?bar/s]

In [9]:
stats

Start                     2000-08-30 00:00...
End                       2021-03-11 00:00...
Duration                   7498 days 01:00:00
Exposure Time [%]                    55.01166
Equity Final [$]                  30122.70358
Equity Peak [$]                   35105.10397
Return [%]                          201.22704
Buy & Hold Return [%]               540.97505
Return (Ann.) [%]                     5.54614
Volatility (Ann.) [%]                13.58666
CAGR [%]                              3.77555
Sharpe Ratio                           0.4082
Sortino Ratio                         0.60863
Calmar Ratio                          0.18738
Alpha [%]                           -88.37439
Beta                                  0.53533
Max. Drawdown [%]                   -29.59842
Avg. Drawdown [%]                    -4.03438
Max. Drawdown Duration     3258 days 00:00:00
Avg. Drawdown Duration      112 days 00:00:00
# Trades                                   77
Win Rate [%]                      

In [10]:
bt_oss = Backtest(oss_data, EMA, cash = 10000)

In [12]:
stats = bt_oss.run()

Backtest.run:   0%|          | 0/1222 [00:00<?, ?bar/s]

In [13]:
stats

Start                     2021-03-12 00:00...
End                       2026-04-23 00:00...
Duration                   1867 days 23:00:00
Exposure Time [%]                    62.62626
Equity Final [$]                  19989.60254
Equity Peak [$]                   22515.60254
Return [%]                           99.89603
Buy & Hold Return [%]               152.81652
Return (Ann.) [%]                    14.52458
Volatility (Ann.) [%]                16.43866
CAGR [%]                              9.79447
Sharpe Ratio                          0.88356
Sortino Ratio                          1.4084
Calmar Ratio                          1.13916
Alpha [%]                            -4.25965
Beta                                  0.68157
Max. Drawdown [%]                   -12.75026
Avg. Drawdown [%]                    -2.76556
Max. Drawdown Duration      309 days 00:00:00
Avg. Drawdown Duration       35 days 00:00:00
# Trades                                   18
Win Rate [%]                      

In [14]:
bt.plot()

GridPlot(id='p1366', ...)

In [15]:
bt_oss.plot()

GridPlot(id='p1757', ...)